# Diabetics

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from scipy.stats.mstats import winsorize

# dataset loading
df = pd.read_csv("diabetes.csv")



# missing values and summary statistics
basic_info = df.info()
missing_values = df.isnull().sum()
summary_stats = df.describe(include='all')

missing_values, summary_stats

# Separazione dei casi con glyhb mancante
df_missing_glyhb = df[df['glyhb'].isnull()].copy()
df = df[~df['glyhb'].isnull()].copy()  # Rimozione dei missing da df centrale


# Conversione categoriche a 'category' dtype
categorical_cols = ['gender', 'location', 'frame']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# Rename problematic column names
df = df.rename(columns={
    'stab.glu': 'stab_glu',})

# Calcolo della diagnosi (glyhb > 7.0)
df['diagnosis'] = (df['glyhb'] > 7.0).astype('category')

# Calcolo del BMI
df['bmi'] = (df['weight'] * 0.453592) / ((df['height'] * 0.0254) ** 2)

# Sostituzione dei valori mancanti di bp.2s e bp.2d con bp.1s e bp.1d
df['bp.2s_filled'] = df['bp.2s'].fillna(df['bp.1s'])
df['bp.2d_filled'] = df['bp.2d'].fillna(df['bp.1d'])

# Calcolo delle medie pressorie
df['bp_s_mean'] = ((df['bp.1s'] + df['bp.2s_filled']) / 2).round()
df['bp_d_mean'] = ((df['bp.1d'] + df['bp.2d_filled']) / 2).round()

# Calcolo Waist-Hip Ratio
df['whr'] = df['waist'] / df['hip']

# Rimozione colonne intermedie
df.drop(columns=['bp.2s_filled', 'bp.2d_filled', 'bp.1s', 'bp.1d', 'bp.2s', 'bp.2d'], inplace=True)


# Filtro: tiene solo le righe complete tranne in variabili rilevanti
excluded = ['location', 'height', 'frame', 'gender', 'waist', 'time.ppn']
df_clean = df[df.drop(columns=excluded).notnull().all(axis=1)].copy()

# Subset finale per modellazione
df_model = df_clean[[
    'chol',
    'age',
    'stab_glu',
    'hdl',
    'ratio',
    'bmi',
    'whr',
    'bp_s_mean',
    'bp_d_mean',
    'diagnosis'
]].copy()

# Verifica finale
print("Shape df_model:", df_model.shape)
print("Missing values in df_model:\n", df_model.isnull().sum())

df_model.info()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Crea la variabile binaria di outcome
df['diabetes'] = np.where(df['glyhb'] > 7.0, 1, 0)

# Variabili continue da plottare
continuous_vars = [
    'age', 'weight', 'height', 'waist', 'hip',
    'ratio', 'hdl', 'chol', 'stab_glu', 'glyhb',
    'bp_s_mean', 'bp_d_mean'
]

# Palette di default di seaborn: il primo colore è il blu, il secondo l’arancione
palette = sns.color_palette()

fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(15, 20))
axes = axes.flatten()

for i, var in enumerate(continuous_vars):
    # Istogramma con hue
    sns.histplot(
        data=df, x=var, hue='diabetes', kde=True,
        ax=axes[i], element='step', common_norm=False,
        palette=palette
    )

    # Calcola le mediane nelle due sottopopolazioni
    median_no_diab = df.loc[df['diabetes'] == 0, var].median()
    median_diab    = df.loc[df['diabetes'] == 1, var].median()

    # Linea verticale per i non diabetici (blu)
    axes[i].axvline(
        median_no_diab, color=palette[0], linestyle='--',
        linewidth=2, label='Mediana non diabetici' if i == 0 else ""
    )
    # Linea verticale per i diabetici (arancione)
    axes[i].axvline(
        median_diab, color=palette[1], linestyle='--',
        linewidth=2, label='Mediana diabetici' if i == 0 else ""
    )

    axes[i].set_title(f'Distribuzione di {var} per stato diabetico')

# Un’unica legenda globale (al primo subplot l’etichetta è stata impostata)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.97])  # lascia spazio per la legenda globale
plt.show()

In [ ]:
# RIMUOVIAMO GLI OUTLIERS E GLI NA
df_model_complete = df_model.dropna()

# Separate predictors and outcome
X = df_model_complete.drop(columns='diagnosis')
y = df_model_complete['diagnosis']

def winsorize_column(col):
    return pd.Series(winsorize(col, limits=[0.01, 0.01]).data, index=col.index)

# Apply winsorization (on already complete data)
# X_winsorized = X.apply(winsorize_column)




# Recombine into a cleaned DataFrame
df_model_cleaned = pd.concat([X, y.reset_index(drop=True)], axis=1)
df_model_cleaned.dropna(inplace=True)


# Confirm consistency
print("Final shape:", df_model_cleaned.shape)
print("Missing values per column:\n", df_model_cleaned.isnull().sum())


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
import numpy as np
import pandas as pd

# Prepare data
X = df_model_cleaned.drop(columns='diagnosis')
y = df_model_cleaned['diagnosis'].astype(int)

# Function to compute OR and CI using bootstrapping
def compute_or_ci(X, y, n_bootstraps=1000, alpha=0.05):
    or_vals = []
    for _ in range(n_bootstraps):
        X_b, y_b = resample(X, y)
        model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000)
        model.fit(X_b, y_b)
        or_vals.append(np.exp(model.coef_[0][0]))
    lower = np.percentile(or_vals, 100 * alpha / 2)
    upper = np.percentile(or_vals, 100 * (1 - alpha / 2))
    return np.mean(or_vals), lower, upper

# Loop through each predictor
crude_or_results = []
for feature in X.columns:
    Xi = X[[feature]].values
    or_val, lower_ci, upper_ci = compute_or_ci(Xi, y)
    crude_or_results.append({
        'Variable': feature,
        'Crude OR': or_val,
        '95% CI Lower': lower_ci,
        '95% CI Upper': upper_ci
    })

crude_or_df = pd.DataFrame(crude_or_results)
print("Univariable (Crude) ORs:")
print(crude_or_df)


In [ ]:
# Full model with all predictors
adjusted_or_results = []
for i, feature in enumerate(X.columns):
    or_vals = []
    for _ in range(1000):
        X_b, y_b = resample(X, y)
        model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000)
        model.fit(X_b, y_b)
        or_vals.append(np.exp(model.coef_[0][i]))
    lower = np.percentile(or_vals, 2.5)
    upper = np.percentile(or_vals, 97.5)
    mean_or = np.mean(or_vals)
    adjusted_or_results.append({
        'Variable': feature,
        'Adjusted OR': mean_or,
        '95% CI Lower': lower,
        '95% CI Upper': upper
    })

adjusted_or_df = pd.DataFrame(adjusted_or_results)
print("\nMultivariable (Adjusted) ORs:")
print(adjusted_or_df)


# Plotting
crude_or_df['Type'] = 'Crude'
adjusted_or_df['Type'] = 'Adjusted'
crude_or_df = crude_or_df.rename(columns={'Crude OR': 'OR'})
adjusted_or_df = adjusted_or_df.rename(columns={'Adjusted OR': 'OR'})
or_combined_df = pd.concat([crude_or_df, adjusted_or_df])

plt.figure(figsize=(10, 6))
for i, (var, group) in enumerate(or_combined_df.groupby('Variable')):
    for j, row in group.iterrows():
        x = row['OR']
        ci_low = row['95% CI Lower']
        ci_up = row['95% CI Upper']
        plt.errorbar(x, i + (0.15 if row['Type'] == 'Crude' else -0.15),
                     xerr=[[x - ci_low], [ci_up - x]],
                     fmt='o', label=row['Type'] if i == 0 else "", 
                     capsize=4, color='blue' if row['Type'] == 'Crude' else 'green')

plt.axvline(1, linestyle='--', color='gray')
plt.yticks(range(len(or_combined_df['Variable'].unique())), or_combined_df['Variable'].unique())
plt.xlabel("Odds Ratio (log scale)")
plt.xscale('log')
plt.title("Crude vs Adjusted Odds Ratios with 95% Confidence Intervals")
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()


# IC

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Verifica il dataset (usa quello che hai definito prima)
X = df_model.drop(columns='diagnosis')  # Cambia se necessario
y = df_model['diagnosis'].astype(int)

print(f"Dataset shape: {X.shape}")
print(f"Target distribution: {y.value_counts()}")

def compute_or_ci(X, y, n_bootstraps=1000, alpha=0.05):
    """Calcola OR e CI con bootstrap"""
    or_vals = []
    
    for _ in range(n_bootstraps):  # CORREZIONE: _ invece di *
        X_b, y_b = resample(X, y, random_state=None)
        
        # Controllo per evitare casi degeneri
        if len(np.unique(y_b)) < 2:
            continue
            
        model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000)
        
        try:
            model.fit(X_b, y_b)
            or_vals.append(np.exp(model.coef_[0][0]))
        except:
            continue
    
    if len(or_vals) == 0:
        return np.nan, np.nan, np.nan
    
    lower = np.percentile(or_vals, 100 * alpha / 2)
    upper = np.percentile(or_vals, 100 * (1 - alpha / 2))
    return np.mean(or_vals), lower, upper

# 1. CRUDE ORs (modelli univariabili)
print("=== CALCOLO OR CRUDE ===")
crude_or_results = []

for feature in X.columns:
    print(f"Analizzando {feature}...")
    X_i = X[[feature]]
    or_val, lower_ci, upper_ci = compute_or_ci(X_i, y)
    
    crude_or_results.append({
        'Variable': feature,
        'Crude_OR': or_val,
        'CI_Lower': lower_ci,
        'CI_Upper': upper_ci
    })

crude_or_df = pd.DataFrame(crude_or_results)
print("\nUnivariable (Crude) ORs:")
print(crude_or_df.round(3))

# 2. ADJUSTED ORs (modello multivariabile)
print("\n=== CALCOLO OR AGGIUSTATE ===")
adjusted_or_results = []

for i, feature in enumerate(X.columns):
    print(f"Analizzando {feature} (posizione {i})...")
    or_vals = []
    
    for _ in range(1000):
        X_b, y_b = resample(X, y, random_state=None)
        
        if len(np.unique(y_b)) < 2:
            continue
            
        model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000)
        
        try:
            model.fit(X_b, y_b)
            or_vals.append(np.exp(model.coef_[0][i]))
        except:
            continue
    
    if len(or_vals) > 0:
        lower = np.percentile(or_vals, 2.5)
        upper = np.percentile(or_vals, 97.5)
        mean_or = np.mean(or_vals)
    else:
        lower = upper = mean_or = np.nan
    
    adjusted_or_results.append({
        'Variable': feature,
        'Adjusted_OR': mean_or,
        'CI_Lower': lower,
        'CI_Upper': upper
    })

adjusted_or_df = pd.DataFrame(adjusted_or_results)
print("\nMultivariable (Adjusted) ORs:")
print(adjusted_or_df.round(3))

# Risk Factors

In [ ]:
# Combina crude e adjusted in un'unica tabella
results_combined = crude_or_df.merge(
    adjusted_or_df, 
    on='Variable', 
    suffixes=('_crude', '_adj')
)

# Formato per pubblicazione
results_combined['Crude_OR_CI'] = results_combined.apply(
    lambda row: f"{row['Crude_OR']:.2f} ({row['CI_Lower_crude']:.2f}-{row['CI_Upper_crude']:.2f})", 
    axis=1
)

results_combined['Adjusted_OR_CI'] = results_combined.apply(
    lambda row: f"{row['Adjusted_OR']:.2f} ({row['CI_Lower_adj']:.2f}-{row['CI_Upper_adj']:.2f})", 
    axis=1
)

publication_table = results_combined[['Variable', 'Crude_OR_CI', 'Adjusted_OR_CI']]
print("\n=== TABELLA PER PUBBLICAZIONE ===")
print(publication_table)

In [ ]:
# Identifica i fattori di rischio significativi
print("\n=== INTERPRETAZIONE EPIDEMIOLOGICA ===")

for _, row in results_combined.iterrows():
    var = row['Variable']
    crude_or = row['Crude_OR']
    adj_or = row['Adjusted_OR']
    
    # Check se CI include 1.0
    crude_significant = not (row['CI_Lower_crude'] <= 1.0 <= row['CI_Upper_crude'])
    adj_significant = not (row['CI_Lower_adj'] <= 1.0 <= row['CI_Upper_adj'])
    
    print(f"\n{var}:")
    print(f"  Crude OR: {crude_or:.2f} ({'Significativo' if crude_significant else 'Non significativo'})")
    print(f"  Adjusted OR: {adj_or:.2f} ({'Significativo' if adj_significant else 'Non significativo'})")
    
    # Interpretazione del cambiamento
    if not np.isnan(crude_or) and not np.isnan(adj_or):
        change = abs(crude_or - adj_or) / crude_or * 100
        if change > 10:
            print(f"  CONFOUNDING IMPORTANTE: cambio del {change:.1f}%")

# Modelli

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# SPLIT STRATIFICATO (mantiene stessa prevalenza)
X_final = df_model_cleaned.drop(columns='diagnosis')
y_final = df_model_cleaned['diagnosis'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, 
    test_size=0.2, 
    stratify=y_final,
    random_state=42
)

print("=== VERIFICA STRATIFICAZIONE ===")
print(f"Dataset completo: {len(y_final)} obs, prevalenza: {y_final.mean():.3f}")
print(f"Training set: {len(y_train)} obs, prevalenza: {y_train.mean():.3f}")
print(f"Test set: {len(y_test)} obs, prevalenza: {y_test.mean():.3f}")
print(f"Differenza prevalenza: {abs(y_train.mean() - y_test.mean()):.4f}")

# SALVA i set per non contaminarli
print("\n⚠  IMPORTANTE: Il test set NON verrà più toccato fino alla validazione finale!")

## Predictive models with interactions (Minimal, Metabolic, Complete)

This section adds three logistic regression models—**Minimal**, **Metabolic**, and **Complete**—enhanced with a small set of clinically sensible interaction terms:

* `age × chol`
* `age × hdl`
* `bmi × bp_s_mean`
* `age × bmi`

Workflow highlights  
* **Train/test split**: 80 % / 20 % &nbsp;·&nbsp; seed = 123  
* **CV**: 15‑fold stratified ROC‑AUC  
* **Performance on test set**: AUC, confusion‑matrix, sensitivity, specificity with **bootstrap 95 % CIs** (1000 repliche)  
* **Calibration** curve (reliability plot) & Brier score

> *Nota:* il codice assume che il *DataFrame* pre‑processato `df_model_cleaned` (con outcome binario `diagnosis`) sia già disponibile dalle celle precedenti.  
> In caso contrario lancia un errore che ricorda di eseguire prima il pre‑processing.

In [ ]:

# %% Predictive models (balanced, scaled) with optimal threshold
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

RANDOM_STATE = 123

# ---------------------------------------------------------------
# Check dataset already prepared
# ---------------------------------------------------------------
try:
    data = df_model_cleaned.copy()
except NameError as e:
    raise RuntimeError('df_model_cleaned not found – run preprocessing first.') from e

# ---------------------------------------------------------------
# Feature sets & interactions
# ---------------------------------------------------------------
minimal_features   = ['age', 'stab_glu']
metabolic_features = ['age', 'stab_glu', 'hdl', 'chol']
complete_features  = ['age', 'stab_glu', 'hdl', 'chol', 'bmi', 'bp_s_mean']

interaction_pairs = [
    ('age', 'chol'),
    ('age', 'hdl'),
    ('bmi', 'bp_s_mean'),
    ('age', 'bmi')
]

def add_interactions(df, pairs):
    out = df.copy()
    for a, b in pairs:
        out[f'{a}_x_{b}'] = out[a] * out[b]
    return out

data = add_interactions(data, interaction_pairs)

# ---------------------------------------------------------------
# Train / test split (stratified)
# ---------------------------------------------------------------
X = data.drop(columns='diagnosis')
y = data['diagnosis'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

# ---------------------------------------------------------------
# Helper: compute optimal threshold on TRAIN by Youden J index
# ---------------------------------------------------------------
def optimal_threshold(model, X_val, y_val):
    y_prob_val = model.predict_proba(X_val)[:, 1]
    fpr, tpr, thr = roc_curve(y_val, y_prob_val)
    J = tpr - fpr
    best_idx = np.argmax(J)
    return thr[best_idx]

# ---------------------------------------------------------------
# Models dictionary
# ---------------------------------------------------------------
models = {
    'Minimal'  : minimal_features,
    'Metabolic': metabolic_features + ['age_x_chol', 'age_x_hdl'],
    'Complete' : complete_features + ['age_x_chol', 'age_x_hdl', 'bmi_x_bp_s_mean', 'age_x_bmi']
}

cv = StratifiedKFold(n_splits=15, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, feats in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=1000, solver='liblinear', class_weight='balanced'))
    ])
    # Cross‑val AUC on TRAIN
    cv_auc = cross_val_score(pipe, X_train[feats], y_train, cv=cv, scoring='roc_auc')
    # Fit on full TRAIN
    pipe.fit(X_train[feats], y_train)
    # Determine optimal threshold on TRAIN
    thr_opt = optimal_threshold(pipe, X_train[feats], y_train)

    # Test predictions
    y_prob_test = pipe.predict_proba(X_test[feats])[:, 1]
    y_pred_test = (y_prob_test >= thr_opt).astype(int)
    cm = confusion_matrix(y_test, y_pred_test)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0
    spec = tn / (tn + fp) if (tn + fp) else 0

    # Store
    results[name] = {
        'features'     : feats,
        'model'        : pipe,
        'threshold'    : thr_opt,
        'cv_auc_mean'  : cv_auc.mean(),
        'cv_auc_std'   : cv_auc.std(ddof=1),
        'test_auc'     : roc_auc_score(y_test, y_prob_test),
        'test_cm'      : cm,
        'test_sens'    : sens,
        'test_spec'    : spec,
        'calibration'  : calibration_curve(y_test, y_prob_test, n_bins=10, strategy='uniform')
    }

# ---------------------------------------------------------------
# Bootstrap 95 % CI on TEST
# ---------------------------------------------------------------
n_boot = 1000
rng = np.random.RandomState(RANDOM_STATE)

for name, info in results.items():
    feats = info['features']
    model = info['model']
    thr   = info['threshold']

    boot_auc, boot_sens, boot_spec = [], [], []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_test), len(y_test))
        y_true_bs = y_test.iloc[idx].values
        X_bs      = X_test.iloc[idx][feats]
        y_prob_bs = model.predict_proba(X_bs)[:, 1]
        y_pred_bs = (y_prob_bs >= thr).astype(int)

        boot_auc.append(roc_auc_score(y_true_bs, y_prob_bs))
        tn, fp, fn, tp = confusion_matrix(y_true_bs, y_pred_bs).ravel()
        boot_sens.append(tp / (tp + fn) if (tp + fn) else np.nan)
        boot_spec.append(tn / (tn + fp) if (tn + fp) else np.nan)

    info['ci_auc']  = (np.nanpercentile(boot_auc, 2.5),  np.nanpercentile(boot_auc, 97.5))
    info['ci_sens'] = (np.nanpercentile(boot_sens, 2.5), np.nanpercentile(boot_sens, 97.5))
    info['ci_spec'] = (np.nanpercentile(boot_spec, 2.5), np.nanpercentile(boot_spec, 97.5))

# ---------------------------------------------------------------
# Summary table
# ---------------------------------------------------------------
summary = pd.DataFrame([
    {
        'Model'              : k,
        'CV AUC (mean±SD)'   : f"{v['cv_auc_mean']:.3f} ± {v['cv_auc_std']:.3f}",
        'Threshold*'         : f"{v['threshold']:.2f}",
        'Test AUC (95% CI)'  : f"{v['test_auc']:.3f} ({v['ci_auc'][0]:.3f}-{v['ci_auc'][1]:.3f})",
        'Sensitivity (95% CI)': f"{v['test_sens']:.3f} ({v['ci_sens'][0]:.3f}-{v['ci_sens'][1]:.3f})",
        'Specificity (95% CI)': f"{v['test_spec']:.3f} ({v['ci_spec'][0]:.3f}-{v['ci_spec'][1]:.3f})"
    }
    for k, v in results.items()
])

display(summary.style.set_caption("*Threshold selected on training set via Youden J index"))

# ---------------------------------------------------------------
# Calibration curves
# ---------------------------------------------------------------
plt.figure()
for name, info in results.items():
    prob_true, prob_pred = info['calibration']
    plt.plot(prob_pred, prob_true, label=name)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('Predicted probability')
plt.ylabel('Observed proportion')
plt.title('Calibration curves (test set)')
plt.legend()
plt.show()
